# Phase 2 & 3: Staff-Level Workflow Modeling & Mixed-Effects Analysis

Models individual nurse beacon-derived workflow features using hierarchical blocks (`M1` care load, `M2` + hospital events, `M3` + subjective busyness) and quantifies individual heterogeneity via Random-Intercept (`ICC`) and Random-Slope Mixed-Effects models.

In [ ]:
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().resolve().parent))
from src.config_loader import load_config
from src.stats_compat import smf, chi2_sf

cfg = load_config()
PATHS = cfg["resolved_paths"]
ANALYSIS_CFG = cfg["analysis"]
HOURS = list(ANALYSIS_CFG["day_shift_hours"])
DAY_SHIFT_TYPE = ANALYSIS_CFG["day_shift_type"]
FLOORS = [str(f) for f in ANALYSIS_CFG["floors"]]
COLORS_FLOOR = {str(k): str(v) for k, v in ANALYSIS_CFG["floor_colors"].items()}
HOSPITAL_VARS = list(ANALYSIS_CFG["hospital_vars"])
BEACON_COLS = list(ANALYSIS_CFG["beacon_feature_cols"])
SKILL_MAP = {int(k): str(v) for k, v in ANALYSIS_CFG["skill_map"].items()}
SKILL_ORDER = list(ANALYSIS_CFG["skill_order"])
SKILL_COLORS = {str(k): str(v) for k, v in ANALYSIS_CFG["skill_colors"].items()}

## 1. Data Preparation

In [ ]:
beacon = pd.read_csv(PATHS["hourly_features"])
beacon["shift_date"] = pd.to_datetime(beacon["shift_date"])

m_shift = pd.read_csv(PATHS["m_shift"])
day_shift_ids = set(m_shift[m_shift["shift_type"] == DAY_SHIFT_TYPE]["shift_id"].dropna().astype(int))

beacon_day = beacon[
    beacon["shift_id"].isin(day_shift_ids)
    & (beacon["hour"] >= HOURS[0])
    & (beacon["hour"] <= HOURS[-1])
].copy()

staff_beacon = (
    beacon_day.groupby(["user_id", "shift_date", "floor"])
    .agg(working_hours=("hour", "count"), **{col: (col, "sum") for col in BEACON_COLS})
    .reset_index()
)
staff_beacon["user_num"] = staff_beacon["user_id"].str.extract(r"(\d+)").astype(int)
staff_beacon["floor"] = staff_beacon["floor"].astype(int).astype(str)
print(f"Staff-beacon records: {staff_beacon.shape}")

In [ ]:
care = pd.read_csv(PATHS["staff_patient_care_level"])
care = care[care["shift_type"] == DAY_SHIFT_TYPE].copy()
care["date"] = pd.to_datetime(care["date"])
care["user_id"] = care["user_id"].astype(str).str.strip()
care = care[care["user_id"].str.match(r"^User\d+$", na=False)]
care["user_num"] = care["user_id"].str.extract(r"(\d+)").astype(int)
care["floor"] = care["ward"].str.extract(r"(\d+)").astype(str)

care_level_cols = [c for c in care.columns if c.startswith("care_level_")]
care["staff_care_load"] = care[care_level_cols].sum(axis=1)

staff_care = care[["user_num", "date", "floor", "staff_care_load"]].copy()
staff_care.columns = ["user_num", "shift_date", "floor", "staff_care_load"]
print(f"Staff-care records: {staff_care.shape}")

In [ ]:
floor_df = pd.read_csv(PATHS["floor_daily_staffing_summary"])
floor_df["floor"] = floor_df["floor"].astype(int).astype(str)
floor_df["shift_date"] = pd.to_datetime(floor_df["shift_date"])
floor_df["is_weekend"] = ((floor_df.get("dow_5", 0) == 1) | (floor_df.get("dow_6", 0) == 1)).astype(int)
floor_df = floor_df[floor_df["is_weekend"] == 0].copy()

floor_data = floor_df[["floor", "shift_date", "busyness"] + HOSPITAL_VARS].copy()

df = (
    staff_beacon
    .merge(staff_care, on=["user_num", "shift_date", "floor"], how="inner")
    .merge(floor_data, on=["floor", "shift_date"], how="inner")
)
print(f"Merged dataset: {df.shape} ({df.user_num.nunique()} users, {df.shift_date.nunique()} dates)")

## 2. Correlation Overview

In [ ]:
corr_cols = ["busyness", "staff_care_load"] + HOSPITAL_VARS + BEACON_COLS
short_names = [
    "busyness", "care_load", "inpatients", "admitted", "nursing_proc",
    "room_time", "NS_time", "hallway", "other_area",
    "travel_dist", "loc_changes", "room_visits", "long_visits",
]
corr = df[corr_cols].corr()
corr.index = short_names
corr.columns = short_names

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax, annot_kws={"size": 8}
)
ax.set_title(f"Staff-Level Correlation Matrix (Weekday Day Shift, n={len(df)})")
plt.tight_layout()
plt.show()

## 3. Hierarchical OLS Models (M1 / M2 / M3)

In [ ]:
print(f"=== Overall Hierarchical OLS (n={len(df)}) ===")
print(f"{'Y':<35} {'M1(care)':>10} {'M2(+hosp)':>10} {'M3(+busy)':>10}")
print("-" * 70)

for ycol in BEACON_COLS:
    sub = df[[ycol, "staff_care_load", "busyness"] + HOSPITAL_VARS].dropna()
    m1 = smf.ols(f"{ycol} ~ staff_care_load", sub).fit()
    m2 = smf.ols(f"{ycol} ~ staff_care_load + " + " + ".join(HOSPITAL_VARS), sub).fit()
    m3 = smf.ols(f"{ycol} ~ staff_care_load + " + " + ".join(HOSPITAL_VARS) + " + busyness", sub).fit()
    print(f"{ycol.replace('f_', ''):<35} {m1.rsquared_adj:>10.4f} {m2.rsquared_adj:>10.4f} {m3.rsquared_adj:>10.4f}")

## 4. Mixed-Effects Model (Random Intercept per Staff)

```
beacon ~ staff_care_load + hospital_vars + (1 | user_id)
```

In [ ]:
df["user_id_str"] = df["user_num"].astype(str)
formula_fe = "staff_care_load + " + " + ".join(HOSPITAL_VARS)

print("=== Mixed Effects Model: beacon ~ care + hospital + (1|user_id) ===")
print(f"{'Y':<35} {'OLS_R2':>8} {'ICC':>8} {'care_coef':>10} {'care_p':>8}")
print("-" * 75)

me_results = []
for ycol in BEACON_COLS:
    ols_res = smf.ols(f"{ycol} ~ {formula_fe}", df).fit()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        me_res = smf.mixedlm(f"{ycol} ~ {formula_fe}", df, groups=df["user_id_str"]).fit(reml=True)

    var_re = float(me_res.cov_re.iloc[0, 0])
    var_resid = me_res.scale
    icc = var_re / (var_re + var_resid) if (var_re + var_resid) > 0 else 0.0
    care_coef = me_res.params["staff_care_load"]
    care_p = me_res.pvalues["staff_care_load"]

    me_results.append({
        "Y": ycol.replace("f_", ""),
        "OLS_AdjR2": ols_res.rsquared_adj,
        "ICC": icc,
        "care_coef": care_coef,
        "care_p": care_p,
        "me_res": me_res,
    })
    print(f"{ycol.replace('f_', ''):<35} {ols_res.rsquared_adj:>8.4f} {icc:>8.3f} {care_coef:>10.2f} {care_p:>8.4f}")

## 5. Robustness: Skill Group × Care Load Interaction

In [ ]:
m_user = pd.read_csv(PATHS["m_user"])
m_user["user_num"] = m_user["user_id"].str.extract(r"(\d+)").astype(int)

df_skill = df.merge(m_user[["user_num", "ladder"]], on="user_num", how="left")
# --- [REDACTED: PROPRIETARY LOGIC] ---
# Conceptually, this step consolidates clinical ladder levels into career-stage
# cohorts and tests a Skill x Care Load interaction specification.
# Below is a simplified baseline main-effects comparison preserving the output table schema.
df_skill["skill_group"] = df_skill["ladder"].map(SKILL_MAP)
df_skill = df_skill.dropna(subset=["skill_group"]).copy()
df_skill["skill_group"] = pd.Categorical(df_skill["skill_group"], categories=SKILL_ORDER, ordered=False)

print(f"Records with valid skill tier: {len(df_skill)}")
print(f"{'Y':<35} {'M2_AdjR2':>10} {'Inter_AdjR2':>13} {'Delta':>8} {'Inter_F_p':>10}")
print("-" * 80)

hospital_rhs = " + ".join(HOSPITAL_VARS)
for ycol in BEACON_COLS:
    m_base = smf.ols(f"{ycol} ~ staff_care_load + {hospital_rhs}", df_skill).fit()
    m_inter = smf.ols(
        f"{ycol} ~ C(skill_group) + staff_care_load + {hospital_rhs}",
        df_skill,
    ).fit()
    inter_p = 0.5000
    delta = m_inter.rsquared_adj - m_base.rsquared_adj
    sig = ""
    print(f"{ycol.replace('f_', ''):<35} {m_base.rsquared_adj:>10.4f} {m_inter.rsquared_adj:>13.4f} {delta:>+8.4f} {inter_p:>10.4f} {sig}")

## 6. Robustness: Random-Slope Mixed-Effects Model

In [ ]:
print("=== Random Slope ME: beacon ~ care + hospital + (1 + staff_care_load | user_id) ===")
print(f"{'Y':<35} {'RI_LL':>10} {'RS_LL':>10} {'LR_chi2':>10} {'LR_p':>10} {'RS_converged':>14}")
print("-" * 95)

for ycol in BEACON_COLS:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            m_ri = smf.mixedlm(f"{ycol} ~ {formula_fe}", df, groups=df["user_id_str"]).fit(reml=False)
            ll_ri = m_ri.llf
        except Exception:
            ll_ri = np.nan
        try:
            m_rs = smf.mixedlm(
                f"{ycol} ~ {formula_fe}", df, groups=df["user_id_str"], re_formula="1 + staff_care_load"
            ).fit(reml=False)
            ll_rs = m_rs.llf
            rs_conv = m_rs.converged
        except Exception:
            ll_rs = np.nan
            rs_conv = False

    if not (np.isnan(ll_ri) or np.isnan(ll_rs)):
        lr = 2 * (ll_rs - ll_ri)
        lr_p = chi2_sf(max(lr, 0), df=2) if lr > 0 else np.nan
    else:
        lr, lr_p = np.nan, np.nan
    print(f"{ycol.replace('f_', ''):<35} {ll_ri:>10.1f} {ll_rs:>10.1f} {lr:>10.2f} {lr_p:>10.4f} {str(rs_conv):>14}")